In [1]:
from pathlib import Path
import json
import boto3

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
sagemaker_client = aws_session.client("sagemaker")
s3_client = aws_session.client("s3")

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=project_config["deployment"]["job_name"]
)
assert batch_details["TransformJobStatus"] == "Completed"

deployed_model_name = batch_details["ModelName"]
deployed_model = sagemaker_client.describe_model(
    ModelName=deployed_model_name
)
serving_container = deployed_model["PrimaryContainer"]

print("Model used for batch predictions:", deployed_model_name)
print("Serving image:", serving_container["Image"])
print("Serving model artifact:", serving_container.get("ModelDataUrl"))
print(
    "Serving configuration keys:",
    sorted(serving_container.get("Environment", {})),
)
print("\nRegistry setup ready. No new model or job created.")

Model used for batch predictions: aai540-group3-routing-ce725539a788
Serving image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
Serving model artifact: s3://aai540-group3-tickets-390568313988-us-east-1/training-output/aai540-group3-lr-20261004-003353/output/model.tar.gz
Serving configuration keys: ['SAGEMAKER_CONTAINER_LOG_LEVEL', 'SAGEMAKER_MODEL_SERVER_WORKERS', 'SAGEMAKER_PROGRAM', 'SAGEMAKER_REGION', 'SAGEMAKER_SUBMIT_DIRECTORY']

Registry setup ready. No new model or job created.


In [2]:
import hashlib
import tarfile
from io import BytesIO
from urllib.parse import urlparse


def read_current_s3_artifact(uri):
    location = urlparse(uri)
    assert location.scheme == "s3", f"Expected an S3 location: {uri}"

    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
    )

    version_id = response.get("VersionId")
    assert version_id and version_id != "null"

    with response["Body"] as body:
        content = body.read()

    return content, {
        "uri": uri,
        "version_id": version_id,
        "sha256": hashlib.sha256(content).hexdigest(),
    }


# Verify the model artifact used by the serving configuration.
final_model = project_config["final_model"]
assert serving_container["ModelDataUrl"] == final_model["model_uri"]

model_bytes, model_artifact = read_current_s3_artifact(
    serving_container["ModelDataUrl"]
)

assert model_artifact["version_id"] == final_model["model_version_id"]
assert model_artifact["sha256"] == final_model["model_archive_sha256"]

print("Trained model version and fingerprint verified.")

# Verify the separately uploaded inference code.
serving_environment = dict(serving_container["Environment"])
code_uri = serving_environment["SAGEMAKER_SUBMIT_DIRECTORY"]

code_bytes, code_artifact = read_current_s3_artifact(code_uri)
expected_source_hashes = project_config["deployment"][
    "inference_source_sha256"
]

with tarfile.open(fileobj=BytesIO(code_bytes), mode="r:gz") as archive:
    for filename, expected_hash in expected_source_hashes.items():
        matches = [
            member for member in archive.getmembers()
            if member.isfile() and Path(member.name).name == filename
        ]
        assert len(matches) == 1, f"Missing or ambiguous file: {filename}"

        extracted = archive.extractfile(matches[0])
        assert extracted is not None
        with extracted:
            actual_hash = hashlib.sha256(extracted.read()).hexdigest()

        assert actual_hash == expected_hash, f"Code changed: {filename}"
        print("Prediction code verified:", filename)

registry_assets = {
    "source_model_name": deployed_model_name,
    "serving_image": serving_container["Image"],
    "serving_environment": serving_environment,
    "model_artifact": model_artifact,
    "inference_code_artifact": code_artifact,
    "inference_source_sha256": expected_source_hashes,
}

registry_report_dir = PROJECT_ROOT / "reports" / "model_registry"
registry_report_dir.mkdir(parents=True, exist_ok=True)

(registry_report_dir / "verified_assets.json").write_text(
    json.dumps(registry_assets, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("\nVerified model and inference-code details saved.")

Trained model version and fingerprint verified.
Prediction code verified: inference.py
Prediction code verified: requirements.txt

Verified model and inference-code details saved.


In [3]:
import math

test_metrics = project_config["final_model"]["test_metrics"]

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    value = float(test_metrics[metric])
    assert math.isfinite(value) and 0.0 <= value <= 1.0

registry_evaluation = {
    "evaluation_split": "test",
    "evaluated_tickets": project_config["final_model"]["test_rows"],
    "training_job_name": project_config["final_model"]["training_job_name"],
    "model_selection_metric": "validation_macro_f1",
    "evaluation_dataset": project_config["final_model"]["test_dataset"],
    "multiclass_classification_metrics": {
        metric: {"value": float(test_metrics[metric])}
        for metric in ["accuracy", "macro_f1", "weighted_f1"]
    },
    "limitations": [
        "Current accuracy is insufficient to establish readiness for "
        "fully automatic ticket routing.",
        "Performance differs across queues and languages.",
        "Small test samples limit conclusions for Spanish, French, "
        "and Portuguese.",
    ],
}

evaluation_bytes = json.dumps(
    registry_evaluation, indent=2, allow_nan=False
).encode("utf-8")

evaluation_sha256 = hashlib.sha256(evaluation_bytes).hexdigest()
evaluation_key = (
    f"registry/evaluations/{evaluation_sha256}/evaluation.json"
)

uploaded = s3_client.put_object(
    Bucket=PROJECT_BUCKET,
    Key=evaluation_key,
    Body=evaluation_bytes,
    ContentType="application/json",
)

evaluation_version = uploaded.get("VersionId")
assert evaluation_version and evaluation_version != "null"

evaluation_artifact = {
    "uri": f"s3://{PROJECT_BUCKET}/{evaluation_key}",
    "version_id": evaluation_version,
    "sha256": evaluation_sha256,
}

(registry_report_dir / "evaluation.json").write_bytes(evaluation_bytes)

registry_config = project_config.setdefault("model_registry", {})
registry_config["verified_assets"] = registry_assets
registry_config["evaluation_report"] = evaluation_artifact

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Registry evaluation report saved locally and in S3.")
print("Evaluation split:", registry_evaluation["evaluation_split"])
print("Evaluated tickets:", registry_evaluation["evaluated_tickets"])

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    print(f"{metric}: {test_metrics[metric]:.4f}")

print("Ready to create the model registry group and version.")

Registry evaluation report saved locally and in S3.
Evaluation split: test
Evaluated tickets: 4428
accuracy: 0.4634
macro_f1: 0.4464
weighted_f1: 0.4653
Ready to create the model registry group and version.


In [4]:
import uuid
from botocore.exceptions import ClientError

model_group_name = "aai540-group3-ticket-routing"

# Reuse the group if it already exists.
try:
    group_details = sagemaker_client.describe_model_package_group(
        ModelPackageGroupName=model_group_name
    )
except ClientError as error:
    error_code = error.response["Error"]["Code"]
    message = error.response["Error"]["Message"].lower()
    missing_group = (
        error_code in {"ResourceNotFound", "ResourceNotFoundException"}
        or (
            error_code == "ValidationException"
            and any(text in message for text in [
                "does not exist", "could not find", "cannot find", "not found"
            ])
        )
    )
    if not missing_group:
        raise

    group_details = sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_group_name,
        ModelPackageGroupDescription=(
            "Group 3 multilingual ticket routing models for 10 queues."
        ),
    )

print("Registry group ready:", model_group_name)

registration_request = {
    "ModelPackageGroupName": model_group_name,
    "ModelPackageDescription": (
        "Tested TF-IDF and Logistic Regression ticket-routing model. "
        "Includes the serving code used for successful batch predictions. "
        "Course demonstration; requires review before operational use."
    ),
    "ModelApprovalStatus": "PendingManualApproval",
    "InferenceSpecification": {
        "Containers": [{
            "Image": registry_assets["serving_image"],
            "ModelDataUrl": registry_assets["model_artifact"]["uri"],
            "Environment": registry_assets["serving_environment"],
        }],
        "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
        "SupportedContentTypes": ["application/jsonlines"],
        "SupportedResponseMIMETypes": ["application/jsonlines"],
    },
    "ModelMetrics": {
        "ModelQuality": {
            "Statistics": {
                "ContentType": "application/json",
                "S3Uri": evaluation_artifact["uri"],
            }
        }
    },
    "CustomerMetadataProperties": {
        "training_job": final_model["training_job_name"],
        "dataset_sha256": project_config["dataset_sha256"],
        "model_sha256": registry_assets["model_artifact"]["sha256"],
        "model_s3_version": registry_assets["model_artifact"]["version_id"],
        "inference_code_sha256": registry_assets[
            "inference_code_artifact"
        ]["sha256"],
        "inference_code_s3_version": registry_assets[
            "inference_code_artifact"
        ]["version_id"],
        "evaluation_sha256": evaluation_artifact["sha256"],
        "evaluation_s3_version": evaluation_artifact["version_id"],
        "selection_metric": "validation_macro_f1",
        "test_accuracy": str(test_metrics["accuracy"]),
        "test_macro_f1": str(test_metrics["macro_f1"]),
        "test_weighted_f1": str(test_metrics["weighted_f1"]),
    },
}

# A stable token protects against duplicate registration on API retries.
registration_request["ClientToken"] = str(uuid.uuid5(
    uuid.NAMESPACE_URL,
    json.dumps(registration_request, sort_keys=True),
))

(registry_report_dir / "registration_request.json").write_text(
    json.dumps(registration_request, indent=2),
    encoding="utf-8",
)

registry_config = project_config["model_registry"]

if registry_config.get("model_package_arn"):
    model_package_arn = registry_config["model_package_arn"]
    print("Using the previously registered model version.")
else:
    registration = sagemaker_client.create_model_package(
        **registration_request
    )
    model_package_arn = registration["ModelPackageArn"]

    # Save the ARN immediately so subsequent runs reuse this version.
    registry_config.update({
        "model_package_group_name": model_group_name,
        "model_package_arn": model_package_arn,
        "registration_client_token": registration_request["ClientToken"],
    })
    config_path.write_text(
        json.dumps(project_config, indent=2),
        encoding="utf-8",
    )

package_details = sagemaker_client.describe_model_package(
    ModelPackageName=model_package_arn
)

registry_config.update({
    "model_package_version": package_details["ModelPackageVersion"],
    "status": package_details["ModelPackageStatus"],
    "approval_status": package_details["ModelApprovalStatus"],
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8",
)

print("Model version:", registry_config["model_package_version"])
print("Registration status:", registry_config["status"])
print("Approval status:", registry_config["approval_status"])
print("Model package ARN:", model_package_arn)

Registry group ready: aai540-group3-ticket-routing
Model version: 1
Registration status: Completed
Approval status: PendingManualApproval
Model package ARN: arn:aws:sagemaker:us-east-1:390568313988:model-package/aai540-group3-ticket-routing/1


In [5]:
registered_container = package_details["InferenceSpecification"]["Containers"][0]

assert registered_container["Image"] == registry_assets["serving_image"]
assert registered_container["ModelDataUrl"] == (
    registry_assets["model_artifact"]["uri"]
)

for key, value in registry_assets["serving_environment"].items():
    assert registered_container["Environment"][key] == value

assert package_details["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"] == (
    evaluation_artifact["uri"]
)
assert package_details["ModelPackageStatus"] == "Completed"
assert package_details["ModelApprovalStatus"] == "PendingManualApproval"

registration_summary = {
    "model_package_group": model_group_name,
    "model_package_arn": model_package_arn,
    "model_version": package_details["ModelPackageVersion"],
    "registration_status": package_details["ModelPackageStatus"],
    "approval_status": package_details["ModelApprovalStatus"],
    "serving_image": registered_container["Image"],
    "model_artifact": registry_assets["model_artifact"],
    "inference_code_artifact": registry_assets["inference_code_artifact"],
    "evaluation_report": evaluation_artifact,
    "serving_configuration_verified": True,
}

(registry_report_dir / "registration_summary.json").write_text(
    json.dumps(registration_summary, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("PASS: registered model, serving code configuration, and report match.")
print("Registration summary saved.")

PASS: registered model, serving code configuration, and report match.
Registration summary saved.
